# Milestone 3 · TV5 — Xuất 6 Bảng Parquet & Upload Hugging Face Hub Cá Nhân (Colab CPU)

Notebook hỗ trợ **mỗi thành viên (`tv1`–`tv5`) tự xuất đúng phần dữ liệu thuộc dải arXiv được phân công** (hoặc chọn `all` để xuất toàn bộ nhóm 6) thành **6 bảng Sharded Parquet** và upload **công khai (public)** lên **tài khoản Hugging Face cá nhân** để lấy link nộp bài theo yêu cầu giảng viên.

### Bảng dải arXiv chính thức của Nhóm 6 (`configs/ms1_group6_ranges.json`)
| `MEMBER` | Dải arXiv ID được giao | Số bài trong split | `train` | `valid` | `test` |
| :---: | :---: | :---: | :---: | :---: | :---: |
| `tv1` | `2508.12872 : 2508.17871` | 754 | 750 | 1 | 3 |
| `tv2` | `2506.15788 : 2506.20787` | 407 | 404 | 1 | 2 |
| `tv3` | `2411.08799 : 2411.13798` | 865 | 859 | 4 | 2 |
| `tv4` | `2410.18025 : 2410.23024` | 835 | 832 | 3 | 0 |
| `tv5` | `2410.03025 : 2410.08024` | 835 | 831 | 1 | 3 |
| `all` | Toàn bộ 5 thành viên | 3,696 | 3,676 | 10 | 10 |

### Quy trình 6 bước
1. Mount Drive, tải repo `arxiv-data-science` (`origin/main`), kiểm tra `pyarrow` & `huggingface_hub`.
2. Chọn **`MEMBER`** (`tv1`–`tv5` cho nộp bài cá nhân, hoặc `all`), trỏ vào dữ liệu MS1/MS2 của **chính bạn** + các file chung của nhóm (`partition_manifest.json`, `labels`, `tv4-real-02_ban_giao.zip`).
3. **Gate-check toàn nhóm** tự động trên file điểm TV4 (`valid MRR@5 = 0.221411`, `test MRR@5 = 0.387560`), nạp & lọc 5 bảng nguồn MS1/MS2 theo đúng dải ID của `MEMBER` (hỗ trợ cả `tables` lẫn `ms1_ms2_checkpoints`).
4. Dựng bảng `predictions` cho đúng các bài split thuộc `MEMBER` (giữ nguyên nhãn `partition = train/valid/test` theo split chính thức `7a08e01a...`, xử lý an toàn cả bài `0` references và ca `tv4` có `0` bài `test`).
5. Kiểm định chéo 6 bảng (`validate_six_tables`), xuất **sharded Parquet (`zstd`)**, đọc ngược kiểm tra schema & SHA-256 (`verify_parquet_export`), sinh Dataset Card `README.md` và đồng bộ sang Google Drive.
6. Upload **công khai (`HF_PRIVATE = False`)** lên tài khoản Hugging Face cá nhân (lưu Write Token vào **Colab Secrets** với tên `HF_TOKEN`), xác minh `publicly_accessible = True`, chạy streaming smoke test và in ra **link công khai để nộp bài**.


In [ ]:
# @title 1. Mount Google Drive, tải code arxiv-data-science & kiểm tra thư viện
MOUNT_DRIVE = True  # @param {type:"boolean"}
REPO_URL = "https://github.com/LeoTkiet/arxiv-data-science.git"  # @param {type:"string"}
# Commit trên origin/main (đã gồm PR #32 TV5 + PR #33 TV1 bindings + PR #34 TV3)
CODE_REF = "ec60057f386b34ee1555c1f0ffa60f1719e61a35"  # @param {type:"string"}
REPO_DIR = "/content/arxiv-data-science"

import importlib
import subprocess
import sys
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

for pkg in ("pyarrow", "huggingface_hub"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

repo = Path(REPO_DIR)
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)

if CODE_REF.strip():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", CODE_REF.strip()], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)

CODE_SHA = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()

src_dir = str((repo / "src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
for mod in list(sys.modules):
    if mod == "arxiv_pipeline" or mod.startswith("arxiv_pipeline."):
        sys.modules.pop(mod, None)
importlib.invalidate_caches()

import pyarrow as pa
import huggingface_hub
from arxiv_pipeline.matching.export import (
    TABLE_NAMES,
    export_hf_dataset,
    prepare_published_references,
    upload_to_hf,
    verify_hf_repo_visibility,
)
from arxiv_pipeline.matching.evaluation import evaluate
from arxiv_pipeline.matching.split import assert_no_split_leakage, load_partition_manifest

print("Code commit       :", CODE_SHA)
print("pyarrow           :", pa.__version__)
print("huggingface_hub   :", huggingface_hub.__version__)
print("6 tables contract :", TABLE_NAMES)


In [ ]:
# @title 2. Chọn thành viên (TV1–TV5 hoặc ALL), đường dẫn Drive & repo Hugging Face cá nhân
# QUAN TRỌNG: Mỗi thành viên chọn đúng mã của mình (tv1..tv5) để xuất đúng phần dữ liệu cá nhân!
# Chỉ chọn "all" nếu muốn xuất bộ gộp toàn nhóm 6.
MEMBER = "tv5"  # @param ["tv1", "tv2", "tv3", "tv4", "tv5", "all"]

# Chọn cách nạp 5 bảng nguồn MS1/MS2:
# - "ms1_ms2_checkpoints" : đọc trực tiếp từ thư mục MS1 & MS2 của chính bạn (tự hỗ trợ cả `MS2_RESULTS.zip`!)
# - "tables"              : đọc từ thư mục chứa sẵn 5 bảng papers/references/elements/hierarchy/bibtex (.parquet/.jsonl/.json)
SOURCE_MODE = "ms1_ms2_checkpoints"  # @param ["ms1_ms2_checkpoints", "tables"]
MS1_ROOT = "/content/drive/MyDrive/arxiv_ms1_recovery/tv5/data"  # @param {type:"string"}
MS2_ROOT = "/content/drive/MyDrive/arxiv_ms2_results/ms2-repaired-20261119/full/tv5"  # @param {type:"string"}
SOURCE_TABLES_DIR = "/content/drive/MyDrive/NMKHDL_MS3_Source_Tables"  # @param {type:"string"}

# Phạm vi 5 bảng nguồn của MEMBER:
# - "all_available" : xuất toàn bộ các bài MS1/MS2 hợp lệ thuộc dải của MEMBER (khuyên dùng để nộp đủ phần crawl/parse của mình)
# - "split_only"    : chỉ xuất các bài của MEMBER nằm trong split MS3
EXPORT_SCOPE = "all_available"  # @param ["all_available", "split_only"]
POPULATION_MANIFEST_PATH = "/content/drive/MyDrive/NMKHDL_MS3_T3-05_Auto_Pool/population_manifest.json"  # @param {type:"string"}

# Đầu vào chung của nhóm (TV5 split + TV3 labels + TV4 frozen scores)
PARTITION_MANIFEST_PATH = "/content/drive/MyDrive/NMKHDL_MS3_TV5_Split/20261126T103040Z-c523a49a/partition_manifest.json"  # @param {type:"string"}
MANUAL_LABELS_PATH = "/content/drive/MyDrive/NMKHDL_MS3_T3-04_Gan_nhan_2026-11-22/04_KET_QUA_COLAB/20261124T094434Z-7edcac30/labels.json"  # @param {type:"string"}
AUTO_LABELS_PATH = "/content/drive/MyDrive/NMKHDL_MS3_T3-05_Auto_Pool/auto_labels.json"  # @param {type:"string"}
TV4_SCORES_PATH = "/content/drive/MyDrive/NMKHDL_MS3_TV4/tv4-real-02_ban_giao.zip"  # @param {type:"string"}

# Tham số xuất Parquet
OUTPUT_DRIVE_ROOT = "/content/drive/MyDrive/NMKHDL_MS3_HF_Export"  # @param {type:"string"}
RUN_ID = ""  # @param {type:"string"}
MAX_ROWS_PER_SHARD = 100000  # @param {type:"integer"}
COMPRESSION = "zstd"  # @param ["zstd", "snappy", "gzip"]
ALLOW_EXTRA_SOURCE_COLUMNS = True  # @param {type:"boolean"}

# Gate-check MRR@5 toàn nhóm (từ kết quả kiểm định chính thức TV5 trên tv4-real-02)
EXPECTED_SPLIT_ID = "7a08e01a4975fefcd37ee619319cef72549e99304b5c16e367655b212bc75be1"  # @param {type:"string"}
EXPECTED_VALID_MRR = 0.221411  # @param {type:"number"}
EXPECTED_TEST_MRR = 0.387560  # @param {type:"number"}

# Cấu hình Hugging Face cá nhân (MẶC ĐỊNH TẮT để kiểm tra local trước; nhớ giữ HF_PRIVATE=False để nộp link công khai!)
DO_UPLOAD_HF = False  # @param {type:"boolean"}
HF_REPO_ID = "your-hf-username/hcmus-arxiv-group6-tv5"  # @param {type:"string"}
HF_PRIVATE = False  # @param {type:"boolean"}
HF_COMMIT_MESSAGE = "Upload verified 6-table arXiv MS3 dataset"  # @param {type:"string"}

from datetime import datetime, timezone
import json
import re
import uuid
from pathlib import Path

for name, val in [
    ("PARTITION_MANIFEST_PATH", PARTITION_MANIFEST_PATH),
    ("MANUAL_LABELS_PATH", MANUAL_LABELS_PATH),
    ("AUTO_LABELS_PATH", AUTO_LABELS_PATH),
    ("TV4_SCORES_PATH", TV4_SCORES_PATH),
    ("OUTPUT_DRIVE_ROOT", OUTPUT_DRIVE_ROOT),
]:
    if val.strip().startswith(("http://", "https://")):
        raise ValueError(f"{name} phải là đường dẫn Drive đã mount (/content/drive/...), không phải URL.")

# Đọc dải arXiv ID chính thức của MEMBER từ configs/ms1_group6_ranges.json
ranges_cfg = json.loads((Path(REPO_DIR) / "configs" / "ms1_group6_ranges.json").read_text(encoding="utf-8-sig"))
if MEMBER == "all":
    MEMBER_RANGE = None
else:
    MEMBER_RANGE = str(ranges_cfg["ranges"][int(MEMBER[2:]) - 1]["range"])

resolved_run_id = RUN_ID.strip() or (
    f"{MEMBER}-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
)
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,79}", resolved_run_id):
    raise ValueError(f"RUN_ID không hợp lệ: {resolved_run_id}")

LOCAL_EXPORT_DIR = Path("/content/ms3-hf-export") / resolved_run_id
DRIVE_EXPORT_DIR = Path(OUTPUT_DRIVE_ROOT.strip()) / MEMBER / resolved_run_id

for label, p in [
    ("partition_manifest", Path(PARTITION_MANIFEST_PATH.strip())),
    ("manual_labels", Path(MANUAL_LABELS_PATH.strip())),
    ("tv4_scores", Path(TV4_SCORES_PATH.strip())),
]:
    if not p.exists():
        raise FileNotFoundError(f"Không tìm thấy {label}: {p}")

if AUTO_LABELS_PATH.strip() and not Path(AUTO_LABELS_PATH.strip()).exists():
    raise FileNotFoundError(f"Không tìm thấy auto_labels: {AUTO_LABELS_PATH}")

if SOURCE_MODE == "tables":
    if not Path(SOURCE_TABLES_DIR.strip()).is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục SOURCE_TABLES_DIR: {SOURCE_TABLES_DIR}")
else:
    if not Path(MS1_ROOT.strip()).is_dir():
        raise FileNotFoundError(f"Không tìm thấy MS1_ROOT: {MS1_ROOT}")
    if not Path(MS2_ROOT.strip()).exists():
        raise FileNotFoundError(f"Không tìm thấy MS2_ROOT (thư mục hoặc file MS2_RESULTS.zip): {MS2_ROOT}")

print("MEMBER           :", MEMBER, "| RANGE :", MEMBER_RANGE or "ALL Group 6")
print("RUN_ID           :", resolved_run_id)
print("SOURCE_MODE      :", SOURCE_MODE, "| EXPORT_SCOPE :", EXPORT_SCOPE)
print("LOCAL_EXPORT_DIR :", LOCAL_EXPORT_DIR)
print("DRIVE_EXPORT_DIR :", DRIVE_EXPORT_DIR)
print("DO_UPLOAD_HF     :", DO_UPLOAD_HF, "| HF_REPO_ID :", HF_REPO_ID, "| PRIVATE :", HF_PRIVATE)


In [ ]:
# @title 3. Nạp Split, cắt lát theo MEMBER, Gate-Check toàn nhóm & nạp 5 bảng nguồn MS1/MS2
from collections import defaultdict
import gc
import hashlib
import io
import json
import math
import zipfile
from pathlib import Path
import pyarrow.parquet as pq

from arxiv_pipeline.matching.contracts import (
    ContractError,
    LabelRecord,
    PARTITIONS,
    PartitionManifest,
    RankedPrediction,
    canonical_id,
)
from arxiv_pipeline.matching.io import load_with_provenance

SOURCE_TABLE_NAMES = ("papers", "references", "elements", "hierarchy", "bibtex")

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def _modern_key(pid: str):
    parts = pid.split(".")
    return (parts[0], int(parts[1])) if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit() else None

def _in_member_range(pid: str, range_str: str | None) -> bool:
    if range_str is None:
        return True
    lo_s, hi_s = range_str.split(":")
    lo_k, hi_k, cur_k = _modern_key(canonical_id(lo_s)), _modern_key(canonical_id(hi_s)), _modern_key(canonical_id(pid))
    return lo_k is not None and hi_k is not None and cur_k is not None and lo_k <= cur_k <= hi_k

def _load_labels(path: Path) -> list[LabelRecord]:
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path, "r") as zf:
            cands = [n for n in zf.namelist() if n.endswith("labels.json")]
            if not cands:
                raise ContractError(f"Không tìm thấy *labels.json trong {path}")
            chosen = "auto_labels.json" if "auto_labels.json" in cands else cands[0]
            raw = json.loads(zf.read(chosen).decode("utf-8-sig"))
    elif path.suffix.lower() == ".jsonl":
        raw = [json.loads(l) for l in path.read_text(encoding="utf-8-sig").splitlines() if l.strip()]
    else:
        raw = json.loads(path.read_text(encoding="utf-8-sig"))
    if isinstance(raw, dict):
        for k in ("labels", "records", "rows", "entries"):
            if isinstance(raw.get(k), list):
                raw = raw[k]
                break
    return [LabelRecord(**item) for item in raw]

def _load_one_source_table(root: Path, tname: str) -> list[dict]:
    cands = [
        root / "data" / tname,
        root / tname,
        root / f"{tname}.parquet",
        root / f"{tname}.jsonl",
        root / f"{tname}.json",
    ]
    target = next((p for p in cands if p.exists()), None)
    if target is None:
        raise FileNotFoundError(f"Không tìm thấy bảng nguồn '{tname}' trong {root}")
    rows = []
    if target.is_dir():
        pqs, jls, js = sorted(target.glob("*.parquet")), sorted(target.glob("*.jsonl")), sorted(target.glob("*.json"))
        if pqs:
            for f in pqs:
                rows.extend(pq.read_table(f).to_pylist())
        elif jls:
            for f in jls:
                rows.extend(json.loads(l) for l in f.read_text(encoding="utf-8-sig").splitlines() if l.strip())
        elif js:
            for f in js:
                obj = json.loads(f.read_text(encoding="utf-8-sig"))
                rows.extend(obj if isinstance(obj, list) else [obj])
        else:
            raise ContractError(f"Thư mục {target} rỗng")
    elif target.suffix.lower() == ".parquet":
        rows = pq.read_table(target).to_pylist()
    elif target.suffix.lower() == ".jsonl":
        rows = [json.loads(l) for l in target.read_text(encoding="utf-8-sig").splitlines() if l.strip()]
    else:
        obj = json.loads(target.read_text(encoding="utf-8-sig"))
        rows = obj[tname] if isinstance(obj, dict) and isinstance(obj.get(tname), list) else obj
    print(f"  Nạp {tname:10s}: {len(rows):>10,} dòng ({target.name})")
    return rows

# 1) Split toàn nhóm + cắt lát theo MEMBER
full_manifest = load_partition_manifest(Path(PARTITION_MANIFEST_PATH.strip()))
assert_no_split_leakage(full_manifest)
if EXPECTED_SPLIT_ID.strip() and full_manifest.split_id != EXPECTED_SPLIT_ID.strip():
    raise ContractError(f"split_id không khớp: {full_manifest.split_id} != {EXPECTED_SPLIT_ID}")

if MEMBER == "all":
    member_manifest = full_manifest
else:
    m_rows = [r for r in full_manifest.rows if _in_member_range(r.arxiv_id, MEMBER_RANGE)]
    member_manifest = PartitionManifest(
        seed=full_manifest.seed,
        group_size=full_manifest.group_size,
        population_snapshot_id=full_manifest.population_snapshot_id,
        label_snapshot_id=full_manifest.label_snapshot_id,
        manual_publication_ids=sorted(r.arxiv_id for r in m_rows if r.pool == "manual"),
        auto_publication_ids=sorted(r.arxiv_id for r in m_rows if r.pool == "auto"),
        rows=m_rows,
        schema_version=full_manifest.schema_version,
    )

target_split_ids = {r.arxiv_id for r in member_manifest.rows}
print("group split_id     :", full_manifest.split_id)
print(f"member ({MEMBER}) pubs  : {len(target_split_ids):,} bài trong split | counts = {member_manifest.counts}")

all_labels = _load_labels(Path(MANUAL_LABELS_PATH.strip()))
if AUTO_LABELS_PATH.strip():
    all_labels.extend(_load_labels(Path(AUTO_LABELS_PATH.strip())))
print(f"Tổng labels nhóm   : {len(all_labels):,} dòng (supervised={sum(1 for l in all_labels if l.is_supervised):,})")

# 2) Nạp 5 bảng nguồn MS1/MS2 đúng phạm vi MEMBER
if SOURCE_MODE == "tables":
    source_tables = {t: _load_one_source_table(Path(SOURCE_TABLES_DIR.strip()), t) for t in SOURCE_TABLE_NAMES}
    if MEMBER_RANGE is not None or EXPORT_SCOPE == "split_only":
        c_cache = {}
        def _keep(raw_pid):
            s = raw_pid if isinstance(raw_pid, str) else str(raw_pid)
            pid = c_cache.get(s)
            if pid is None:
                pid = canonical_id(s)
                c_cache[s] = pid
            if MEMBER_RANGE is not None and not _in_member_range(pid, MEMBER_RANGE):
                return False
            if EXPORT_SCOPE == "split_only" and pid not in target_split_ids:
                return False
            return True
        for t in SOURCE_TABLE_NAMES:
            before = len(source_tables[t])
            source_tables[t] = [r for r in source_tables[t] if _keep(r["arxiv_id"])]
            if len(source_tables[t]) != before:
                print(f"  Lọc {t:10s} ({MEMBER}/{EXPORT_SCOPE}): {before:,} -> {len(source_tables[t]):,}")
    source_prov = {"source_mode": "tables", "source_dir": SOURCE_TABLES_DIR.strip(), "export_scope": EXPORT_SCOPE}
else:
    if EXPORT_SCOPE == "split_only" or not POPULATION_MANIFEST_PATH.strip():
        target_ids = sorted(target_split_ids)
    else:
        pop_raw = json.loads(Path(POPULATION_MANIFEST_PATH.strip()).read_text(encoding="utf-8-sig"))
        raw_ids = pop_raw.get("publication_ids") or pop_raw.get("population_ids") or pop_raw
        target_ids = sorted({canonical_id(p) for p in raw_ids if _in_member_range(p, MEMBER_RANGE)})
    # Tự động nhận diện MS1_ROOT (có hoặc chưa có /data) & giải nén MS2_RESULTS.zip ra SSD local Colab cho nhanh
    ms1_p = Path(MS1_ROOT.strip())
    if not (ms1_p / "state").is_dir() and (ms1_p / "data" / "state").is_dir():
        ms1_p = ms1_p / "data"
        print("Auto-resolved MS1_ROOT ->", ms1_p)
    ms2_p = Path(MS2_ROOT.strip())
    if ms2_p.is_dir():
        if (ms2_p / "state").is_dir():
            resolved_ms2 = ms2_p
        elif MEMBER != "all" and (ms2_p / MEMBER / "state").is_dir():
            resolved_ms2 = ms2_p / MEMBER
        elif (ms2_p / "MS2_RESULTS.zip").is_file():
            ms2_p = ms2_p / "MS2_RESULTS.zip"
            resolved_ms2 = None
        else:
            raise FileNotFoundError(f"Không tìm thấy 'state/' hoặc 'MS2_RESULTS.zip' bên trong {ms2_p}")
    else:
        resolved_ms2 = None
    if resolved_ms2 is None:
        if not (ms2_p.is_file() and ms2_p.suffix.lower() == ".zip"):
            raise FileNotFoundError(f"MS2_ROOT không hợp lệ: {ms2_p}")
        unpack_dir = Path("/content/ms2_unpacked") / f"{MEMBER}_{ms2_p.stat().st_size}"
        done_marker = unpack_dir / ".extracted_ok"
        if not done_marker.is_file():
            unpack_dir.mkdir(parents=True, exist_ok=True)
            print(f"Đang giải nén {ms2_p} ra SSD local {unpack_dir} (nhanh hơn đọc trực tiếp trên Drive rất nhiều)...")
            with zipfile.ZipFile(ms2_p, "r") as zf:
                zf.extractall(unpack_dir)
            done_marker.write_text("ok\n", encoding="utf-8")
        else:
            print("Dùng lại bản MS2 đã giải nén trên SSD local:", unpack_dir)
        if (unpack_dir / "state").is_dir():
            resolved_ms2 = unpack_dir
        elif MEMBER != "all" and (unpack_dir / MEMBER / "state").is_dir():
            resolved_ms2 = unpack_dir / MEMBER
        else:
            sub = next((c for c in sorted(unpack_dir.iterdir()) if c.is_dir() and (c / "state").is_dir()), None)
            if sub is None:
                raise ContractError(f"Giải nén {ms2_p} xong nhưng không thấy thư mục 'state/' bên trong {unpack_dir}")
            resolved_ms2 = sub
    print("Resolved MS1_ROOT :", ms1_p)
    print("Resolved MS2_ROOT :", resolved_ms2)
    ctx = {"ms1_root": str(ms1_p), "ms2_root": str(resolved_ms2), "fixture_mode": False}
    source_tables = {t: [] for t in SOURCE_TABLE_NAMES}
    not_ready = []
    for idx, pid in enumerate(target_ids, start=1):
        loaded = load_with_provenance(pid, ctx, include_export_tables=True)
        if loaded.bundle.readiness != "ready" or loaded.source_tables is None:
            not_ready.append(f"{pid}:{loaded.bundle.readiness}")
            continue
        for t in SOURCE_TABLE_NAMES:
            source_tables[t].extend(loaded.source_tables[t])
        if idx % 500 == 0 or idx == len(target_ids):
            print(f"  Checkpoint loaded ({MEMBER}): {idx:,}/{len(target_ids):,} (not_ready={len(not_ready)})")
    if not_ready:
        raise ContractError(f"Có {len(not_ready)} bài chưa ready ở MS1/MS2 của {MEMBER}, ví dụ: {not_ready[:5]}")
    source_prov = {"source_mode": "ms1_ms2_checkpoints", "n_papers": len(target_ids)}

published_refs, ref_stats = prepare_published_references(source_tables["references"])
source_tables["references"] = published_refs
source_prov["ref_stats"] = ref_stats
print("prepare_published_references:", ref_stats)
gc.collect()


In [ ]:
# @title 4. Gate-Check toàn nhóm & Dựng bảng `predictions` riêng cho MEMBER
from collections import defaultdict
import io
import json
import math
import zipfile
import pyarrow.parquet as pq

tv4_path = Path(TV4_SCORES_PATH.strip())
tv4_prov = {"tv4_source": str(tv4_path)}

handles = {}
if tv4_path.is_file() and tv4_path.suffix.lower() == ".zip":
    tv4_prov["tv4_zip_sha256"] = _sha256_file(tv4_path)
    with zipfile.ZipFile(tv4_path, "r") as zf:
        names = zf.namelist()
        frozen_cands = [n for n in names if n.endswith("/frozen.json")]
        if frozen_cands:
            frozen = json.loads(zf.read(frozen_cands[0]).decode("utf-8"))
            tv4_prov.update({
                "model_id": frozen.get("model_id"),
                "model_kind": frozen.get("kind"),
                "model_sha256": frozen.get("model_sha256"),
                "split_id": frozen.get("split_id"),
                "feature_schema_id": frozen.get("feature_schema_id"),
            })
        for part in PARTITIONS:
            matches = [n for n in names if n.endswith(f"scores/{part}/scores.parquet")]
            if not matches:
                raise ContractError(f"Thiếu scores/{part}/scores.parquet trong {tv4_path}")
            handles[part] = pq.ParquetFile(io.BytesIO(zf.read(matches[0])))
else:
    frozen_f = tv4_path / "frozen.json"
    if frozen_f.is_file():
        frozen = json.loads(frozen_f.read_text(encoding="utf-8"))
        tv4_prov.update({
            "model_id": frozen.get("model_id"),
            "model_kind": frozen.get("kind"),
            "model_sha256": frozen.get("model_sha256"),
            "split_id": frozen.get("split_id"),
            "feature_schema_id": frozen.get("feature_schema_id"),
        })
    for part in PARTITIONS:
        p = tv4_path / "scores" / part / "scores.parquet"
        if not p.is_file():
            raise ContractError(f"Thiếu {p}")
        handles[part] = pq.ParquetFile(p)

if tv4_prov.get("split_id") and tv4_prov["split_id"] != full_manifest.split_id:
    raise ContractError(f"TV4 split_id ({tv4_prov['split_id']}) != group manifest ({full_manifest.split_id})")

model_name = str(tv4_prov.get("model_id") or "tv4_model")

def _stream_build_predictions(target_m, full_m, parts, bib_map=None, ref_map=None, verbose=True):
    target_part_by_pub = {r.arxiv_id: r.partition for r in target_m.rows if r.partition in parts}
    full_part_by_pub = {r.arxiv_id: r.partition for r in full_m.rows}
    sup_truth = {}
    lbl_q_by_pub = defaultdict(set)
    for lbl in all_labels:
        if lbl.arxiv_id in target_part_by_pub:
            lbl_q_by_pub[lbl.arxiv_id].add(lbl.citation_key)
            if lbl.is_supervised:
                sup_truth[lbl.key] = lbl.groundtruth_arxiv_id
    canon_cache = {}
    def _canon(raw):
        s = raw if isinstance(raw, str) else str(raw)
        res = canon_cache.get(s)
        if res is None:
            res = canonical_id(s)
            canon_cache[s] = res
        return res
    top5_by_q = {}
    cnt_by_q = defaultdict(int)
    scored_q_by_pub = defaultdict(set)
    kept_pairs = 0
    for part in parts:
        pf = handles[part]
        exp_target = {pid for pid, p in target_part_by_pub.items() if p == part}
        exp_full = {pid for pid, p in full_part_by_pub.items() if p == part}
        seen_refs_by_q = defaultdict(set)
        part_kept = 0
        part_total = 0
        for batch in pf.iter_batches(batch_size=250_000, columns=["arxiv_id", "citation_key", "ref_arxiv_id", "score"]):
            for raw_pid, ckey, raw_rid, raw_sc in zip(
                batch.column(0).to_pylist(), batch.column(1).to_pylist(),
                batch.column(2).to_pylist(), batch.column(3).to_pylist(),
            ):
                part_total += 1
                pid = _canon(raw_pid)
                if pid not in exp_full:
                    raise ContractError(f"Score bài {pid} không thuộc partition '{part}'")
                if pid not in exp_target:
                    continue
                rid = _canon(raw_rid)
                sc = float(raw_sc)
                if not math.isfinite(sc):
                    raise ContractError(f"Điểm không hữu hạn cho ({pid}, {ckey}, {rid})")
                if ref_map is not None and rid not in ref_map.get(pid, ()):
                    raise ContractError(f"Ứng viên {rid} không nằm trong references của {pid}")
                qkey = (pid, ckey)
                qs = seen_refs_by_q[qkey]
                if rid in qs:
                    raise ContractError(f"Trùng cặp score {(pid, ckey, rid)}")
                qs.add(rid)
                scored_q_by_pub[pid].add(ckey)
                cnt_by_q[qkey] += 1
                item = (-sc, rid)
                cur = top5_by_q.get(qkey)
                if cur is None:
                    top5_by_q[qkey] = [item]
                elif len(cur) < 5:
                    cur.append(item)
                    cur.sort()
                elif item < cur[-1]:
                    cur[-1] = item
                    cur.sort()
                part_kept += 1
        seen_refs_by_q.clear()
        kept_pairs += part_kept
        if verbose:
            print(f"  TV4 scores [{part:5s}]: giữ {part_kept:,} / {part_total:,} cặp ({len(exp_target):,} bài của {MEMBER})")
    out = []
    for pid in sorted(target_part_by_pub):
        part = target_part_by_pub[pid]
        qset = bib_map.get(pid, set()) if bib_map is not None else (scored_q_by_pub.get(pid, set()) | lbl_q_by_pub.get(pid, set()))
        if not qset:
            raise ContractError(f"Bài split {pid} không có dòng nào trong bảng bibtex")
        if bib_map is not None:
            extra_q = scored_q_by_pub.get(pid, set()) - qset
            if extra_q:
                raise ContractError(f"TV4 scores có {len(extra_q)} citation_key không có trong bibtex của {pid}")
        exp_refs = len(ref_map.get(pid, set())) if ref_map is not None else None
        for ckey in sorted(qset):
            qkey = (pid, ckey)
            if exp_refs is not None and cnt_by_q.get(qkey, 0) != exp_refs:
                raise ContractError(f"Thiếu/thừa điểm cho {qkey}: nhận {cnt_by_q.get(qkey, 0)}, cần {exp_refs}")
            top_ids = [rid for _, rid in top5_by_q.get(qkey, [])]
            padded = top_ids + [None] * (5 - len(top_ids))
            out.append(RankedPrediction(
                arxiv_id=pid, partition=part, citation_key=ckey,
                groundtruth_arxiv_id=sup_truth.get(qkey),
                cand_1=padded[0], cand_2=padded[1], cand_3=padded[2], cand_4=padded[3], cand_5=padded[4],
            ))
    return out

# 1) Gate-check nhanh toàn nhóm trên valid & test (< 1 giây)
gate_preds = _stream_build_predictions(full_manifest, full_manifest, ("valid", "test"), verbose=False)
group_valid_rep = evaluate(gate_preds, all_labels, full_manifest, partition="valid", model_name=model_name)
group_test_rep = evaluate(gate_preds, all_labels, full_manifest, partition="test", model_name=model_name)
group_valid_mrr = group_valid_rep.metrics["mrr_at_5"]
group_test_mrr = group_test_rep.metrics["mrr_at_5"]
print(f"GROUP GATE-CHECK VALID MRR@5 : {group_valid_mrr:.6f} (expected {EXPECTED_VALID_MRR:.6f})")
print(f"GROUP GATE-CHECK TEST  MRR@5 : {group_test_mrr:.6f} (expected {EXPECTED_TEST_MRR:.6f})")
assert abs(group_valid_mrr - EXPECTED_VALID_MRR) < 1e-5, "Group VALID MRR@5 lệch kỳ vọng!"
assert abs(group_test_mrr - EXPECTED_TEST_MRR) < 1e-5, "Group TEST MRR@5 lệch kỳ vọng!"
print("GROUP GATE-CHECK PASS!")

# 2) Dựng bảng predictions đúng phạm vi MEMBER
bibtex_queries_by_pub = defaultdict(set)
for r in source_tables["bibtex"]:
    bibtex_queries_by_pub[canonical_id(str(r["arxiv_id"]))].add(str(r["citation_key"]))
refs_by_pub = defaultdict(set)
for r in source_tables["references"]:
    refs_by_pub[canonical_id(str(r["arxiv_id"]))].add(canonical_id(str(r["ref_arxiv_id"])))

predictions = _stream_build_predictions(
    member_manifest, full_manifest, PARTITIONS,
    bib_map=bibtex_queries_by_pub, ref_map=refs_by_pub, verbose=True,
)
print(f"Tổng dòng predictions ({MEMBER}): {len(predictions):,} dòng ({len(target_split_ids):,} bài)")

member_metrics = {}
for p_eval in ("valid", "test"):
    n_p = sum(member_manifest.counts[p_eval].values())
    if n_p > 0:
        mrep = evaluate(predictions, all_labels, member_manifest, partition=p_eval, model_name=model_name)
        member_metrics[f"{p_eval}_mrr_at_5"] = mrep.metrics["mrr_at_5"]
        print(f"  Riêng {MEMBER} ({p_eval:5s}): MRR@5 = {mrep.metrics['mrr_at_5']:.6f} ({n_p} bài)")
    else:
        member_metrics[f"{p_eval}_mrr_at_5"] = None
        print(f"  Riêng {MEMBER} ({p_eval:5s}): N/A (0 bài {p_eval} thuộc dải {MEMBER})")


In [ ]:
# @title 5. Kiểm định chéo 6 bảng, xuất Sharded Parquet, Read-Back Verify & đồng bộ Drive
import shutil
from arxiv_pipeline.common.utils import utc_now

all_six_tables = {**source_tables, "predictions": predictions}
provenance = {
    "exported_at_utc": utc_now(),
    "exporter_role": "TV5",
    "member": MEMBER,
    "member_range": MEMBER_RANGE,
    "export_scope": EXPORT_SCOPE,
    "code_commit": CODE_SHA,
    "group_split_id": full_manifest.split_id,
    "member_split_id": member_manifest.split_id,
    "label_snapshot_id": full_manifest.label_snapshot_id,
    "group_valid_mrr_at_5": group_valid_mrr,
    "group_test_mrr_at_5": group_test_mrr,
    "member_metrics": member_metrics,
    "tv4": tv4_prov,
    "source": source_prov,
}

export_manifest = export_hf_dataset(
    all_six_tables,
    LOCAL_EXPORT_DIR,
    partition_manifest=member_manifest,
    max_rows_per_shard=MAX_ROWS_PER_SHARD,
    compression=COMPRESSION,
    synthetic=False,
    allow_extra_source_columns=ALLOW_EXTRA_SOURCE_COLUMNS,
    provenance=provenance,
    upload=False,
)

# Bổ sung thông tin phạm vi thành viên & benchmark vào Dataset Card README.md (giữ nguyên YAML đầu file)
card_path = LOCAL_EXPORT_DIR / "README.md"
scope_desc = f"Member `{MEMBER.upper()}` (assigned range `{MEMBER_RANGE}`)" if MEMBER != "all" else "Full Group 6 (`TV1`–`TV5` combined)"
extra_md = (
    "\n## Provenance & Scope (HCMUS · Group 6 · Milestone 3)\n\n"
    f"- **Dataset scope:** {scope_desc}\n"
    f"- **Official group `split_id`:** `{full_manifest.split_id}` (`seed=42`, `group_size=5`, `stratify=True`)\n"
    f"- **Included split publications:** `{len(member_manifest.rows):,}` "
    f"(`train={sum(member_manifest.counts['train'].values())}`, "
    f"`valid={sum(member_manifest.counts['valid'].values())}`, "
    f"`test={sum(member_manifest.counts['test'].values())}`)\n"
    f"- **Frozen TV4 ranking model:** `{model_name}`\n"
    f"- **Official group benchmark (`include_zero_in_denominator`):** "
    f"`valid MRR@5 = {group_valid_mrr:.6f}`, `test MRR@5 = {group_test_mrr:.6f}` (100% Rank-1 on matched queries)\n"
)
card_path.write_text(card_path.read_text(encoding="utf-8").rstrip() + "\n" + extra_md, encoding="utf-8")

print(f"=== EXPORT & READ-BACK VERIFICATION ({MEMBER.upper()}): PASS ===")
print("readback_verified   :", export_manifest.readback_verified)
print("ready_for_hf_upload :", export_manifest.ready_for_hf_upload)
print(f"total_rows          : {export_manifest.total_rows:,}")
for tname in TABLE_NAMES:
    ti = export_manifest.tables[tname]
    print(f"  - {tname:12s}: {ti['row_count']:>10,} rows | {ti['shard_count']:>3} shard(s) | {ti['total_bytes'] / (1024*1024):>7.2f} MB")
print("partition_counts    :", export_manifest.partition_counts)

if DRIVE_EXPORT_DIR.exists():
    raise FileExistsError(f"Thư mục đích trên Drive đã tồn tại: {DRIVE_EXPORT_DIR}")
DRIVE_EXPORT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_EXPORT_DIR, DRIVE_EXPORT_DIR)
print("Đã đồng bộ bản xuất sang Google Drive:", DRIVE_EXPORT_DIR)
print("\n--- Preview README.md (Hugging Face Dataset Card) ---")
print(card_path.read_text(encoding="utf-8"))


In [ ]:
# @title 6. Upload công khai lên tài khoản Hugging Face cá nhân & lấy link nộp bài
import getpass
import json
import os

if not DO_UPLOAD_HF:
    print("DO_UPLOAD_HF = False -> Đang dừng ở chế độ kiểm tra cục bộ (DRY-RUN).")
    print("Sau khi kiểm tra kỹ bảng tổng kết & README.md ở Cell 5, đặt HF_REPO_ID = '<username_HF_của_bạn>/<tên_dataset>', bật DO_UPLOAD_HF = True ở Cell 2 và chạy lại Cell 6.")
else:
    if not HF_REPO_ID.strip() or "your-hf-username" in HF_REPO_ID:
        raise ValueError("Hãy điền đúng HF_REPO_ID dưới tài khoản Hugging Face cá nhân của bạn ở Cell 2!")
    if HF_PRIVATE:
        print("CẢNH BÁO: Giảng viên yêu cầu link chia sẻ CÔNG KHAI — nên để HF_PRIVATE = False!")

    hf_token = None
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    if not hf_token:
        hf_token = os.environ.get("HF_TOKEN") or getpass.getpass("Nhập Hugging Face Write Token cá nhân (HF_TOKEN): ")

    upload_res = upload_to_hf(
        LOCAL_EXPORT_DIR,
        HF_REPO_ID.strip(),
        manifest=export_manifest,
        token=hf_token,
        private=HF_PRIVATE,
        commit_message=f"{HF_COMMIT_MESSAGE.strip()} ({MEMBER.upper()})",
    )
    vis = verify_hf_repo_visibility(
        HF_REPO_ID.strip(),
        token=hf_token,
        expected_manifest=export_manifest,
    )
    public_url = f"https://huggingface.co/datasets/{HF_REPO_ID.strip()}"
    print("=== UPLOAD HUGGING FACE THÀNH CÔNG ===")
    print("LINK NỘP BÀI        :", public_url)
    print("publicly_accessible :", vis["publicly_accessible"])
    print("all_six_tables      :", vis["all_six_tables_present"])
    print("Visibility detail   :", json.dumps(vis, indent=2))

    try:
        from datasets import load_dataset
        print("\n--- Streaming Smoke Test (1 dòng / bảng) ---")
        for tname in TABLE_NAMES:
            ds = load_dataset(HF_REPO_ID.strip(), name=tname, split="train", streaming=True, token=hf_token if HF_PRIVATE else None)
            sample = next(iter(ds))
            print(f"  OK [{tname:12s}] keys={list(sample.keys())}")
    except ImportError:
        print("Tip: Cài thêm `pip install datasets` nếu muốn chạy streaming smoke test trực tiếp trên Colab.")
